# Kaggle Hybrid Agent Visualizer, Excel-driven

This notebook loads results from an Excel file instead of embedding data in the notebook.

Expected columns, case-insensitive:
- `Model`
- `depth`
- `move`
- `Kaggle`

Extra columns are preserved and shown in tables when useful.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, clear_output
import ipywidgets as widgets

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 80)


In [2]:
# ---------------------------------------------------------------------
# Excel input
# ---------------------------------------------------------------------
# Put your Excel file next to this notebook, or set an absolute path.
# Example:
# EXCEL_PATH = r"C:/Users/uporabnik/Documents/js/Connect4/kaggle_hybrid_results.xlsx"

EXCEL_PATH = "Selected EVAL_PPO_results.xlsx"
SHEET_NAME = "Hybrid"          
HEADER_ROW = 0          # 0 = first row contains headers

REQUIRED_COLUMNS = ["Model", "depth", "move", "Kaggle"]

def available_excel_sheets(path=EXCEL_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Excel file not found: {path.resolve()}")
    return pd.ExcelFile(path).sheet_names

def _normalize_col_name(c):
    return str(c).strip().lower().replace(" ", "").replace("_", "")

def load_results_excel(path=EXCEL_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Excel file not found: {path.resolve()}\n"
            "Put the file next to this notebook, or update EXCEL_PATH."
        )

    raw = pd.read_excel(path, sheet_name=sheet_name, header=header, engine="openpyxl")
    raw = raw.dropna(how="all").copy()

    # Remove unnamed empty columns that Excel likes to breed in dark corners.
    raw = raw.loc[:, ~raw.columns.astype(str).str.match(r"^Unnamed", case=False)]

    col_map = {_normalize_col_name(c): c for c in raw.columns}
    wanted = {
        "model": "Model",
        "depth": "depth",
        "move": "move",
        "kaggle": "Kaggle",
    }

    rename = {}
    missing = []
    for norm, canonical in wanted.items():
        if norm in col_map:
            rename[col_map[norm]] = canonical
        else:
            missing.append(canonical)

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Found columns: {list(raw.columns)}"
        )

    df = raw.rename(columns=rename).copy()
    df["Model"] = df["Model"].astype(str).str.strip()
    df["depth"] = pd.to_numeric(df["depth"], errors="coerce").astype("Int64")
    df["move"] = pd.to_numeric(df["move"], errors="coerce").astype("Int64")
    df["Kaggle"] = pd.to_numeric(df["Kaggle"], errors="coerce")

    df = df.dropna(subset=["Model", "depth", "move"], how="any").copy()

    df["label"] = df.apply(
        lambda r: f"{r['Model']} | d={int(r['depth'])} | m={int(r['move'])} | K={'' if pd.isna(r['Kaggle']) else int(r['Kaggle'])}",
        axis=1,
    )
    df["series_depth"] = df["Model"].astype(str) + " | d=" + df["depth"].astype(str)
    df["series_move"] = df["Model"].astype(str) + " | m=" + df["move"].astype(str)
    return df.reset_index(drop=True)

print("Available sheets:", available_excel_sheets(EXCEL_PATH))
df = load_results_excel(EXCEL_PATH, SHEET_NAME, HEADER_ROW)
display(df.head(30))
print(f"Loaded {len(df)} rows from {EXCEL_PATH!r}.")


Available sheets: ['Main', 'Bases', 'Extended evals', 'Hybrid']


,Model,depth,move,Kaggle,label,series_depth,series_move
0,"none, LA only",7,1,682,"none, LA only | d=7 | m=1 | K=682","none, LA only | d=7","none, LA only | m=1"
1,"none, LA only",9,1,644,"none, LA only | d=9 | m=1 | K=644","none, LA only | d=9","none, LA only | m=1"
2,2004,7,18,708,2004 | d=7 | m=18 | K=708,2004 | d=7,2004 | m=18
3,2004,9,18,788,2004 | d=9 | m=18 | K=788,2004 | d=9,2004 | m=18
4,2004,11,18,757,2004 | d=11 | m=18 | K=757,2004 | d=11,2004 | m=18
5,2004,13,18,751,2004 | d=13 | m=18 | K=751,2004 | d=13,2004 | m=18
6,2004,7,20,666,2004 | d=7 | m=20 | K=666,2004 | d=7,2004 | m=20
7,2004,9,20,734,2004 | d=9 | m=20 | K=734,2004 | d=9,2004 | m=20
8,2004,11,20,796,2004 | d=11 | m=20 | K=796,2004 | d=11,2004 | m=20
9,2004,13,20,834,2004 | d=13 | m=20 | K=834,2004 | d=13,2004 | m=20


Loaded 28 rows from 'Selected EVAL_PPO_results.xlsx'.


In [3]:
# ---------------------------------------------------------------------
# Reload helper, useful after editing/saving the Excel file
# ---------------------------------------------------------------------

def reload_data(redraw_table=True, clear_existing_plots=True):
    global df

    df = load_results_excel(EXCEL_PATH, SHEET_NAME, HEADER_ROW)

    # These are defined in later cells, so guard them.
    if "refresh_widget_options" in globals():
        refresh_widget_options()

    if redraw_table and "update_table" in globals():
        update_table()

    # Do not auto-draw plots on reload.
    # Just clear the old plots so stale charts do not lie to us.
    if clear_existing_plots and "plot_out" in globals():
        with plot_out:
            clear_output(wait=True)
            print("Plots cleared. Press Draw plots to regenerate.")

    return df


reload_button = widgets.Button(
    description="Reload Excel",
    button_style="success",
    icon="refresh",
)

reload_out = widgets.Output()


def _on_reload_clicked(_):
    with reload_out:
        clear_output(wait=True)
        try:
            reload_data(redraw_table=True, clear_existing_plots=True)
            print(f"Reloaded {len(df)} rows from {EXCEL_PATH!r}.")
        except Exception as e:
            print(type(e).__name__ + ":", e)


reload_button.on_click(_on_reload_clicked)

display(widgets.HBox([reload_button]), reload_out)

Output()

In [4]:
# ---------------------------------------------------------------------
# Interactive filters + styled table
# ---------------------------------------------------------------------

def _sorted_unique(s):
    return sorted([x for x in s.dropna().unique().tolist()], key=lambda x: str(x))

def _int_unique(s):
    return sorted([int(x) for x in s.dropna().unique().tolist()])

model_filter = widgets.SelectMultiple(description="Model", layout=widgets.Layout(width="260px"))
depth_filter = widgets.SelectMultiple(description="Depth", layout=widgets.Layout(width="180px"))

move_filter = widgets.IntRangeSlider(
    description="Move",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

score_filter = widgets.IntRangeSlider(
    description="Kaggle",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

include_unscored = widgets.Checkbox(value=True, description="include unscored")
sort_by = widgets.Dropdown(options=["Kaggle", "move", "depth", "Model"], value="Kaggle", description="Sort by")
ascending = widgets.Checkbox(value=False, description="ascending")

table_out = widgets.Output()


def _safe_set_int_range_slider(slider, lo, hi):
    """
    Safely update IntRangeSlider bounds.

    ipywidgets validates every assignment immediately, so setting min before max
    can fail when the new min is larger than the old max.
    """
    lo = int(lo)
    hi = int(hi)

    if lo > hi:
        lo, hi = hi, lo

    # Expand first, then tighten.
    if hi > slider.max:
        slider.max = hi
    if lo < slider.min:
        slider.min = lo

    # Now both bounds can safely contain the desired value.
    slider.value = (lo, hi)

    # Final tightening.
    slider.min = lo
    slider.max = hi
    slider.value = (lo, hi)


def refresh_widget_options():
    models = _sorted_unique(df["Model"])
    depths = _int_unique(df["depth"])
    moves = _int_unique(df["move"])

    model_filter.options = models
    model_filter.value = tuple(models)

    depth_filter.options = depths
    depth_filter.value = tuple(depths)

    if moves:
        _safe_set_int_range_slider(move_filter, min(moves), max(moves))
    else:
        _safe_set_int_range_slider(move_filter, 0, 1)

    if df["Kaggle"].notna().any():
        score_min = int(np.nanmin(df["Kaggle"]))
        score_max = int(np.nanmax(df["Kaggle"]))
    else:
        score_min, score_max = 0, 1000

    _safe_set_int_range_slider(score_filter, score_min, score_max)


def get_filtered_df():
    f = df.copy()

    f = f[f["Model"].isin(list(model_filter.value))]
    f = f[f["depth"].astype(float).isin([float(x) for x in depth_filter.value])]

    f = f[
        (f["move"].astype(float) >= move_filter.value[0]) &
        (f["move"].astype(float) <= move_filter.value[1])
    ]

    scored = f["Kaggle"].between(
        score_filter.value[0],
        score_filter.value[1],
        inclusive="both",
    )

    f = f[scored | f["Kaggle"].isna()] if include_unscored.value else f[scored]

    if sort_by.value == "Kaggle":
        f = f.sort_values(
            ["Kaggle", "depth", "move"],
            ascending=[ascending.value, True, True],
            na_position="last",
        )
    else:
        f = f.sort_values(sort_by.value, ascending=ascending.value, na_position="last")

    return f.reset_index(drop=True)


def update_table(*_):
    with table_out:
        clear_output(wait=True)

        f = get_filtered_df()
        print(f"Rows: {len(f)}")

        show_cols = [
            c for c in f.columns
            if c not in ["label", "series_depth", "series_move"]
        ]

        display(
            f[show_cols]
            .style
            .background_gradient(subset=["Kaggle"], cmap="YlGn", axis=None)
            .format({"Kaggle": "{:.0f}"}, na_rep="")
        )


# Important: refresh first, observe after.
refresh_widget_options()

for w in [model_filter, depth_filter, move_filter, score_filter, include_unscored, sort_by, ascending]:
    w.observe(update_table, names="value")


display(
    widgets.VBox([
        widgets.HBox([model_filter, depth_filter]),
        widgets.HBox([move_filter, score_filter]),
        widgets.HBox([include_unscored, sort_by, ascending]),
    ]),
    table_out,
)

update_table()

Output()

In [5]:
# ---------------------------------------------------------------------
# Interactive Plotly plot options, HTML-rendered plots on button click
# ---------------------------------------------------------------------

import plotly.express as px
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

plot_choices = [
    "Scatter: Kaggle vs move",
    "Line: Kaggle vs move by depth",
    "Line: Kaggle vs depth by move",
    "Bar: configurations ranked",
    "Heatmap: depth x move",
]

plot_select = widgets.SelectMultiple(
    options=plot_choices,
    value=("Scatter: Kaggle vs move", "Heatmap: depth x move"),
    description="Plots",
    rows=len(plot_choices),
    layout=widgets.Layout(width="340px"),
)

text_labels = widgets.Checkbox(value=True, description="point labels")
connect_lines = widgets.Checkbox(value=True, description="connect lines")

plot_height = widgets.IntSlider(
    value=520,
    min=320,
    max=900,
    step=20,
    description="Height",
    continuous_update=False,
)

draw_button = widgets.Button(
    description="Draw plots",
    button_style="success",
    icon="bar-chart",
)

clear_button = widgets.Button(
    description="Clear plots",
    button_style="warning",
    icon="trash",
)

plot_out = widgets.Output()


def _scored(f):
    return f[f["Kaggle"].notna()].copy()


def _point_label(f):
    return f.apply(
        lambda r: f"{r['Model']} d{int(r['depth'])} m{int(r['move'])}",
        axis=1,
    )


def _display_plotly_fig(fig, include_plotlyjs=False):
    """
    More reliable than fig.show() inside widgets.Output().
    First figure should include plotly.js, later figures can reuse it.
    """
    html = fig.to_html(
        full_html=False,
        include_plotlyjs=include_plotlyjs,
        config={
            "displaylogo": False,
            "responsive": True,
        },
    )
    display(HTML(html))


def _make_figures():
    f = _scored(get_filtered_df())

    if f.empty:
        return [], "No scored rows after filtering."

    f = f.copy()
    f["depth_str"] = f["depth"].astype(str)
    f["move_str"] = f["move"].astype(str)
    f["point_label"] = _point_label(f)

    selected = list(plot_select.value)

    if not selected:
        return [], "No plot type selected."

    figs = []

    if "Scatter: Kaggle vs move" in selected:
        fig = px.scatter(
            f,
            x="move",
            y="Kaggle",
            color="depth_str",
            symbol="Model",
            text="point_label" if text_labels.value else None,
            hover_data=[
                c for c in ["Model", "depth", "move", "Kaggle"]
                if c in f.columns
            ],
            title="Kaggle score vs LA takeover move",
        )
        fig.update_traces(textposition="top center")
        fig.update_layout(
            height=plot_height.value,
            xaxis_title="LA takeover move / stones",
            yaxis_title="Kaggle score",
        )
        figs.append(fig)

    if "Line: Kaggle vs move by depth" in selected:
        lf = f.sort_values(["Model", "depth", "move"]).copy()

        fig = px.line(
            lf,
            x="move",
            y="Kaggle",
            color="series_depth",
            markers=True,
            text="point_label" if text_labels.value else None,
            hover_data=["Model", "depth", "move", "Kaggle"],
            title="Kaggle score vs move, grouped by model/depth",
        )
        fig.update_traces(textposition="top center")

        if not connect_lines.value:
            fig.update_traces(
                mode="markers+text" if text_labels.value else "markers"
            )

        fig.update_layout(
            height=plot_height.value,
            xaxis_title="LA takeover move / stones",
            yaxis_title="Kaggle score",
        )
        figs.append(fig)

    if "Line: Kaggle vs depth by move" in selected:
        lf = f.sort_values(["Model", "move", "depth"]).copy()

        fig = px.line(
            lf,
            x="depth",
            y="Kaggle",
            color="series_move",
            markers=True,
            text="point_label" if text_labels.value else None,
            hover_data=["Model", "depth", "move", "Kaggle"],
            title="Kaggle score vs LA depth, grouped by model/move",
        )
        fig.update_traces(textposition="top center")

        if not connect_lines.value:
            fig.update_traces(
                mode="markers+text" if text_labels.value else "markers"
            )

        fig.update_layout(
            height=plot_height.value,
            xaxis_title="LA depth",
            yaxis_title="Kaggle score",
        )
        figs.append(fig)

    if "Bar: configurations ranked" in selected:
        bf = f.sort_values("Kaggle", ascending=False).copy()
        bf["combo"] = bf.apply(
            lambda r: f"{r['Model']} | d{int(r['depth'])} | m{int(r['move'])}",
            axis=1,
        )

        fig = px.bar(
            bf,
            x="combo",
            y="Kaggle",
            color="depth_str",
            text="Kaggle",
            hover_data=["Model", "depth", "move", "Kaggle"],
            title="Configurations ranked by Kaggle score",
        )
        fig.update_traces(
            texttemplate="%{text:.0f}",
            textposition="outside",
        )
        fig.update_layout(
            height=plot_height.value,
            xaxis_title="Configuration",
            yaxis_title="Kaggle score",
            xaxis_tickangle=-35,
        )
        figs.append(fig)

    if "Heatmap: depth x move" in selected:
        hf = f.pivot_table(
            index="depth",
            columns="move",
            values="Kaggle",
            aggfunc="max",
        )

        fig = px.imshow(
            hf,
            text_auto=".0f",
            aspect="auto",
            title="Best Kaggle score heatmap: LA depth x takeover move",
            labels=dict(
                x="LA takeover move / stones",
                y="LA depth",
                color="Kaggle",
            ),
        )
        fig.update_layout(height=plot_height.value)
        figs.append(fig)

    return figs, None


def draw_selected_plots(_=None):
    with plot_out:
        clear_output(wait=True)

        print("Drawing plots...")

        figs, msg = _make_figures()
        if msg is not None:
            print(msg)
            return

        for i, fig in enumerate(figs):
            _display_plotly_fig(fig, include_plotlyjs=(i == 0))

        print(f"Displayed {len(figs)} plot(s).")


def clear_plots(_=None):
    with plot_out:
        clear_output(wait=True)


draw_button.on_click(draw_selected_plots)
clear_button.on_click(clear_plots)

display(
    widgets.VBox([
        widgets.HBox([
            plot_select,
            widgets.VBox([
                text_labels,
                connect_lines,
                plot_height,
                widgets.HBox([draw_button, clear_button]),
            ]),
        ]),
        plot_out,
    ])
)

In [6]:
# ---------------------------------------------------------------------
# Ranking summaries
# ---------------------------------------------------------------------

def show_rankings():
    scored = df[df["Kaggle"].notna()].copy()
    if scored.empty:
        print("No scored rows.")
        return

    print("Top configurations")
    display(
        scored.sort_values("Kaggle", ascending=False)[["Model", "depth", "move", "Kaggle"]]
        .head(30)
        .style.background_gradient(subset=["Kaggle"], cmap="YlGn", axis=None)
        .format({"Kaggle": "{:.0f}"})
    )

    print("Best per model/depth")
    display(
        scored.sort_values("Kaggle", ascending=False)
        .drop_duplicates(["Model", "depth"])[["Model", "depth", "move", "Kaggle"]]
        .sort_values(["Model", "depth"])
        .style.background_gradient(subset=["Kaggle"], cmap="YlGn", axis=None)
        .format({"Kaggle": "{:.0f}"})
    )

    print("Best per model/move")
    display(
        scored.sort_values("Kaggle", ascending=False)
        .drop_duplicates(["Model", "move"])[["Model", "depth", "move", "Kaggle"]]
        .sort_values(["Model", "move"])
        .style.background_gradient(subset=["Kaggle"], cmap="YlGn", axis=None)
        .format({"Kaggle": "{:.0f}"})
    )

show_rankings()


Top configurations


,Model,depth,move,Kaggle
9,2004,13,20,834
12,2004,9,22,802
8,2004,11,20,796
3,2004,9,18,788
11,2004,7,22,761
20,2004,7,26,760
4,2004,11,18,757
5,2004,13,18,751
15,2004,7,24,745
16,2004,8,24,741


Best per model/depth


,Model,depth,move,Kaggle
10,2004,5,22,731
11,2004,7,22,761
16,2004,8,24,741
12,2004,9,22,802
8,2004,11,20,796
9,2004,13,20,834
0,"none, LA only",7,1,682
1,"none, LA only",9,1,644


Best per model/move


,Model,depth,move,Kaggle
3,2004,9,18,788
9,2004,13,20,834
12,2004,9,22,802
15,2004,7,24,745
20,2004,7,26,760
24,2004,7,28,686
25,2004,7,30,719
26,2004,7,32,589
27,2004,7,34,636
0,"none, LA only",7,1,682
